In [24]:
%pip install ucimlrepo

Note: you may need to restart the kernel to use updated packages.


In [25]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
room_occupancy_estimation = fetch_ucirepo(id=864) 
  
# data (as pandas dataframes) 
X = room_occupancy_estimation.data.features 
y = room_occupancy_estimation.data.targets 

# metadata 
print(room_occupancy_estimation.metadata) 
  
# variable information 
print(room_occupancy_estimation.variables) 

{'uci_id': 864, 'name': 'Room Occupancy Estimation', 'repository_url': 'https://archive.ics.uci.edu/dataset/864/room+occupancy+estimation', 'data_url': 'https://archive.ics.uci.edu/static/public/864/data.csv', 'abstract': 'Data set for estimating the precise number of occupants in a room using multiple non-intrusive environmental sensors like temperature, light, sound, CO2 and PIR.', 'area': 'Computer Science', 'tasks': ['Classification'], 'characteristics': ['Multivariate', 'Time-Series'], 'num_instances': 10129, 'num_features': 18, 'feature_types': ['Real'], 'demographics': [], 'target_col': ['Room_Occupancy_Count'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2018, 'last_updated': 'Wed Aug 16 2023', 'dataset_doi': '10.24432/C5P605', 'creators': ['Adarsh Pal Singh', 'Sachin Chaudhari'], 'intro_paper': {'ID': 275, 'type': 'NATIVE', 'title': 'Machine Learning-Based Occupancy Estimation Using Multivariate Sensor Nodes', 'auth

Data Preprocessing

Data Cleaning, Normalization, Feature Engineering

- I don't believe there's any cleaning to do, we're not doing any clustering so I doubt that anything needs to be normalized.
- We do have many features, so possibly collapsing some of the dimensions could be helpful. But how do we decide what gets collapsed and what stays? 

Building the Linear Regression Model
We can try with varying numbers of features.

First set up the data:

In [26]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

X.head()

# changing date and time to something a bit more managable
X['Hour'] = pd.to_datetime(X['Time'], format='%H:%M:%S').dt.hour
X = X.drop(columns=['Date', 'Time'])

LRModel = LinearRegression()
log = {}

Model including all features:

In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

LRModel.fit(X_train, y_train)

predictions = LRModel.predict(X_test)
print('Intercept (β0):', LRModel.intercept_)
print('Coefficients (β_i):', LRModel.coef_)
print('R-squared:', r2_score(y_test, predictions))

log['full'] = r2_score(y_test, predictions)


Intercept (β0): [-8.95764885]
Coefficients (β_i): [[ 1.16281685e-01  1.17845472e-01  8.35644104e-01 -6.90475428e-01
   5.35724724e-03  8.10701821e-04  2.34177077e-03 -3.85726816e-03
   9.56518706e-02  2.37285604e-01 -7.79929633e-02 -3.58311421e-01
   8.34418093e-05  1.92821591e-01  1.83050948e-01  4.15255627e-01
  -3.14904840e-03]]
R-squared: 0.8879846295036886


Now compared to other models:

In [28]:
# time

X_time = X[['Hour']]
X_time_train, X_time_test, y_time_train, y_time_test = train_test_split(
    X_time, y, test_size=0.2, random_state=42)

LRModel_time = LRModel.fit(X_time_train, y_time_train)
predictions = LRModel_time.predict(X_time_test)

log['Time'] = r2_score(y_time_test, predictions)

# temperature

X_temperature = X[['S1_Temp', 'S2_Temp', 'S3_Temp', 'S4_Temp']]
X_temperature_train, X_temperature_test, y_temperature_train, y_temperature_test = train_test_split(
    X_temperature, y, test_size=0.2, random_state=42)

LRModel_temperature = LRModel.fit(X_temperature_train, y_temperature_train)
predictions = LRModel_temperature.predict(X_temperature_test)

log['Temperature'] = r2_score(y_temperature_test, predictions)

# light

X_light = X[['S1_Light', 'S2_Light', 'S3_Light', 'S4_Light']]
X_light_train, X_light_test, y_light_train, y_light_test = train_test_split(
    X_light, y, test_size=0.2, random_state=42)

LRModel_light = LRModel.fit(X_light_train, y_light_train)
predictions = LRModel_light.predict(X_light_test)

log['Light'] = r2_score(y_light_test, predictions)

# sound

X_sound = X[['S1_Sound', 'S2_Sound', 'S3_Sound', 'S4_Sound']]
X_sound_train, X_sound_test, y_sound_train, y_sound_test = train_test_split(
    X_sound, y, test_size=0.2, random_state=42)

LRModel_sound = LRModel.fit(X_sound_train, y_sound_train)
predictions = LRModel_sound.predict(X_sound_test)

log['Sound'] = r2_score(y_sound_test, predictions)

# CO2

X_co2 = X[['S5_CO2', 'S5_CO2_Slope']]
X_co2_train, X_co2_test, y_co2_train, y_co2_test = train_test_split(
    X_co2, y, test_size=0.2, random_state=42)

LRModel_co2 = LRModel.fit(X_co2_train, y_co2_train)
predictions = LRModel_co2.predict(X_co2_test)

log['CO2'] = r2_score(y_co2_test, predictions)

# PIR

X_pir = X[['S6_PIR', 'S7_PIR']]
X_pir_train, X_pir_test, y_pir_train, y_pir_test = train_test_split(
    X_pir, y, test_size=0.2, random_state=42)

LRModel_pir = LRModel.fit(X_pir_train, y_pir_train)
predictions = LRModel_pir.predict(X_pir_test)

log['PIR'] = r2_score(y_pir_test, predictions)

# print results
print(f"{'Model':<20}{'R²':>8}")
print("-" * 28)
for title, r2 in log.items():
    print(f"{title:<20}{r2:>8.3f}")

Model                     R²
----------------------------
full                   0.888
Time                   0.065
Temperature            0.533
Light                  0.812
Sound                  0.391
CO2                    0.723
PIR                    0.552
